> **Prefer Google Colab?** [Open this notebook in Colab](https://colab.research.google.com/github/SpatialTurn/Foundations/blob/main/notebooks/03a_Working_with_Tables_in_pandas.ipynb).
> In Colab, click **File > Save a copy in Drive** first so your changes are kept.

# Working with Tables in pandas

**Series:** Working with Data (notebook 1 of 2)
**Time:** about 25 minutes

**In this notebook you will:**
- Read a CSV file from the web into a DataFrame
- Look at the rows, columns and data types
- Select columns, filter rows and sort
- Add calculated columns and group values

**You should already know:** lists, dictionaries, loops and functions (notebooks 02d to 02f)

> **Before you start:** run each cell in order by clicking it and pressing **Shift + Enter**.

## pandas and DataFrames

**pandas** is the standard Python library for tables. Its main object is the **DataFrame**: rows and
labeled columns, like a spreadsheet, but driven by code so every step can be repeated. A library is loaded
with `import`; `pd` is the nickname everyone uses.

In [ ]:
import pandas as pd

## Read a table from the web

`pd.read_csv()` reads a CSV file straight from its web address into memory; nothing is downloaded by
hand. The file has one row per Indiana county: its name, FIPS code, population in 2012, land area in square
miles, and unemployment rate in 2016.

In [ ]:
url = "https://raw.githubusercontent.com/SpatialTurn/Foundations/main/data/indiana_county_stats.csv"
counties = pd.read_csv(url)
counties.head()          # the first five rows

## First look

In [ ]:
print(counties.shape)        # (rows, columns)
print(counties.columns)      # column names
counties.dtypes              # the type of each column

Notice that `fips` was read as a **number** (`int64`). pandas guesses types from the values. For
identifiers you usually want text; you will see why when joining tables in notebook 04c.

`describe()` gives quick statistics for the numeric columns: a fast way to spot impossible values. It even
averages the FIPS codes, which makes no sense and is another sign they should be text.

In [ ]:
counties.describe().round(1)

## Select columns

One column, in single square brackets, is a **Series** (a single column with labels). A list of columns,
in double square brackets, is a smaller DataFrame.

In [ ]:
print(counties["pop_2012"].sum())       # total population of Indiana in 2012
counties[["name", "unemp_2016"]].head()

## Filter rows

A comparison on a column gives `True` or `False` for every row. Putting it inside square brackets keeps
only the `True` rows. Combine conditions with `&` (and) and `|` (or), with brackets around each part.

In [ ]:
high = counties[counties["unemp_2016"] >= 5.5]
print(len(high), "counties with unemployment of 5.5% or more")
high[["name", "unemp_2016"]]

In [ ]:
small_and_high = counties[(counties["pop_2012"] < 30000) & (counties["unemp_2016"] >= 5)]
small_and_high[["name", "pop_2012", "unemp_2016"]]

## Sort

In [ ]:
counties.sort_values("pop_2012", ascending=False).head(10)[["name", "pop_2012"]]

## Add calculated columns

Math on a column works on every row at once, with no loop needed.

In [ ]:
counties["density"] = counties["pop_2012"] / counties["land_sq_mi"]
counties["land_km2"] = counties["land_sq_mi"] * 2.59
counties[["name", "density", "land_km2"]].round(1).head()

## Group values into classes

`pd.cut()` sorts numbers into classes you define. `value_counts()` then counts each class, and
`groupby()` summarizes another column for each class.

In [ ]:
counties["density_class"] = pd.cut(
    counties["density"],
    bins=[0, 100, 300, 10000],
    labels=["rural", "mixed", "urban"],
)
print(counties["density_class"].value_counts())
counties.groupby("density_class", observed=True)["unemp_2016"].mean().round(2)

## Tables from lists and dictionaries

The patterns from notebook 02d become DataFrames directly. This is handy when you collect values in a loop.

In [ ]:
rows = [
    {"name": "Purdue Memorial Union", "lat": 40.4247, "lon": -86.9107},
    {"name": "Indiana Statehouse", "lat": 39.7684, "lon": -86.1626},
]
places = pd.DataFrame(rows)
places

Saving is one line: `counties.to_csv("counties_with_density.csv", index=False)`. The file appears
in the file browser on the left, next to this notebook; right-click it and choose **Download** to keep a copy.

## Try it yourself

1. Show the five counties with the **lowest** unemployment rate, with their name and rate only.
2. How many counties have more than 100,000 people?
3. What is the average unemployment rate of the urban counties?

<details>
<summary><b>Show a solution</b></summary>

```python
print(counties.sort_values("unemp_2016").head(5)[["name", "unemp_2016"]])
print((counties["pop_2012"] > 100000).sum())     # True counts as 1
print(counties[counties["density_class"] == "urban"]["unemp_2016"].mean())
```

</details>

In [ ]:
# Write your answer here

## Recap

- `pd.read_csv(url)` reads a table from the web into a DataFrame
- `head()`, `shape`, `dtypes` and `describe()` give a first look
- `df["col"]` selects a column; `df[condition]` filters rows; `sort_values()` sorts
- New columns are calculated for every row at once; `pd.cut()` and `groupby()` summarize by class

**Next:** [Plotting with matplotlib](03b_Plotting_with_matplotlib.ipynb)